In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import os
import bs4
import requests
import yfinance as yf

In [2]:
# first scraping wikipedia to get 500 comapnies of S&P500
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
page = requests.get(url, headers=headers)
print(page.status_code)

200


In [3]:
# print(page.content)

In [4]:
soup = bs4.BeautifulSoup(page.content, 'html.parser')
# soup.body

In [5]:
#  syntax: find_all(name, attrs, recursive, string, limit, **kwargs)
all_links = soup.body.find_all(name='a', attrs={'class':'external', 'rel':'nofollow'})
all_links[:6]

[<a class="external text" href="https://www.nyse.com/quote/XNYS:MMM" rel="nofollow">MMM</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:AOS" rel="nofollow">AOS</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:ABT" rel="nofollow">ABT</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:ABBV" rel="nofollow">ABBV</a>,
 <a class="external text" href="https://www.nyse.com/quote/XNYS:ACN" rel="nofollow">ACN</a>,
 <a class="external text" href="https://www.nasdaq.com/market-activity/stocks/adbe" rel="nofollow">ADBE</a>]

In [6]:
import re

def remove_tags(text):
    # r'<.*?>' matches everything between < and > non-greedily
    clean_text = re.sub(r'<.*?>', '', text)
    return clean_text


In [7]:
# only get the scripts name
all_scripts = []
for link in all_links:
    clean = remove_tags(str(link))
    if (len(clean) > 7) or ("." in clean) or (clean == "Q"):
        continue
    all_scripts.append(clean)

In [8]:
len(all_scripts)

500

In [9]:
data = yf.download(
    tickers = all_scripts, 
    period = "1y",      # Use "1y" for one year
    group_by = 'ticker' # Organizes the multi-index by Ticker
)

data.head()

[*********************100%***********************]  500 of 500 completed

1 Failed download:
['AMZN']: TypeError("'NoneType' object is not subscriptable")


Ticker           NCLH                                                  PLTR  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2025-03-07  19.670000  20.100000  19.139999  20.059999  18214000  80.025002   
2025-03-10  19.299999  19.410000  18.290001  18.740000  19677400  82.000000   
2025-03-11  18.600000  19.010000  17.700001  18.770000  19919100  75.785004   
2025-03-12  19.250000  19.610001  18.760000  19.010000  21344300  83.480003   
2025-03-13  19.100000  19.320000  18.209999  18.530001  11902000  82.845001   

Ticker                                                  ...         UNP  \
Price            High        Low      Close     Volume  ...        Open   
Date                                                    ...               
2025-03-07  85.160004  79.150002  84.910004  105377100  ...  240.419699   
2025-03-10  82.690002  74.570000  76.379997  136843800  ...  241.914440   
2025-03-11  80.745003  75.529999  78.050003  109351500  ...  241.748364   
2025-03-12  84.550003  79.860001  83.650002  116525100  ...  231.236298   
2025-03-13  83.739998  78.320000  79.620003  100927200  ...  231.685688   

Ticker                                                         CBOE  \
Price             High         Low       Close   Volume        Open   
Date                                                                  
2025-03-07  244.562008  239.315746  243.565506  2835800  212.871090   
2025-03-10  245.470568  240.449002  242.569000  2420700  213.567440   
2025-03-11  242.373617  231.959237  232.301178  3357900  217.785270   
2025-03-12  232.906889  228.627821  232.017868  1927500  210.225010   
2025-03-13  232.604031  228.139339  228.891586  2399900  206.474717   

Ticker                                                   
Price             High         Low       Close   Volume  
Date                                                     
2025-03-07  217.098867  211.438617  211.886261   933800  
2025-03-10  218.053853  211.816633  217.874802   945200  
2025-03-11  219.476393  209.628149  210.055908  1244900  
2025-03-12  210.225010  201.988305  207.409805  1057200  
2025-03-13  215.169023  206.474717  214.005142  1060500  

[5 rows x 2501 columns]

In [10]:
# Grab only the 'High' values across all tickers
high_df = data.xs('High', axis=1, level=1)
final_df = high_df
final_df

Ticker,NCLH,PLTR,PPG,CCI,DHI,GRMN,A,VTRS,VZ,RVTY,...,VLTO,LUV,HOOD,STE,BLK,JBHT,L,DTE,UNP,CBOE
Date,,,,,,,,,,,,,,,,,,,,,
2025-03-07,20.100000,85.160004,113.467864,93.198089,133.896099,217.000991,127.077039,9.269060,43.239798,117.925602,...,99.813228,28.722814,45.369999,232.749450,933.785801,162.559822,85.583671,128.221843,244.562008,217.098867
2025-03-10,19.410000,82.690002,114.851384,96.108139,136.909240,215.889016,125.638604,9.376139,44.325466,120.049571,...,101.973352,28.282428,41.196999,235.062059,912.964878,162.747742,85.743263,129.566458,245.470568,218.053853
2025-03-11,19.010000,80.745003,112.688410,94.909892,133.648301,216.646738,122.295524,9.164141,41.096522,118.683442,...,100.131771,32.040371,38.349998,229.950485,904.332731,158.929778,84.945280,128.492692,242.373617,219.476393
2025-03-12,19.610001,84.550003,111.129524,92.237579,131.586679,214.471982,122.712167,9.125595,40.141876,114.305857,...,97.324622,30.163083,39.671001,225.930729,904.881508,153.558959,84.626084,128.096087,232.906889,210.225010
2025-03-13,19.320000,83.739998,109.853181,92.009329,128.068070,212.503899,119.121083,9.019595,41.012285,112.122057,...,95.433284,31.058917,38.680000,223.657839,895.054010,152.737974,85.134794,128.666825,232.604031,215.169023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-02,22.879999,147.139999,120.870003,90.129997,156.880005,255.899994,119.400002,15.770000,50.259998,96.400002,...,97.800003,49.020000,79.540001,251.350006,1069.565675,234.449997,112.320000,148.580002,268.140015,305.679993
2026-03-03,21.719999,147.500000,119.139999,90.790001,155.039993,252.320007,119.099998,15.230000,51.099998,96.169998,...,96.279999,48.320000,77.400002,244.589996,1061.082876,231.660004,112.099998,149.429993,265.920013,305.000000
2026-03-04,21.910000,154.520004,120.320000,91.309998,153.250000,253.110001,121.169998,14.970000,51.410000,98.570000,...,95.720001,48.389999,83.849998,245.940002,1057.104973,236.000000,111.800003,150.160004,266.980011,302.549988


In [11]:
final_df.to_csv('snp_500_dailyhigh.csv')

In [12]:
final_df.columns

Index(['NCLH', 'PLTR', 'PPG', 'CCI', 'DHI', 'GRMN', 'A', 'VTRS', 'VZ', 'RVTY',
       ...
       'VLTO', 'LUV', 'HOOD', 'STE', 'BLK', 'JBHT', 'L', 'DTE', 'UNP', 'CBOE'],
      dtype='str', name='Ticker', length=500)